# P2C-64 v3.0 (`WACA_G2`: MPC-PureV64 + SAFE_BND 2-Stage Cascade Global Eval) — Google Colab 実行ノートブック

Egaroucid 公開学習データ（約2,551万局面）および既存の v2.0 Phase-1 ベースライン（`raw_ntuple64.p2c`）を再利用し、**P2C-64 v3.0（`WACA_G2`）** を学習・ビット完全一致監査・純スカラー実戦1,000局対局（Pure PVS / Fair Std-MPC / ゲート掃引 / MPC $\sigma$）するためのワンクリックノートブックです。

### v3.0 主な革新点
- **Tier-1 無劣化分離（0% オーバーヘッド）**：既存の `raw_ntuple64.p2c`（46.92 MiB）を 1 バイトも肥大化させず Stage 1 骨格として共有。Phase 1 再学習は **0 分**（Drive 上の v2.0 成果物を自動再利用）。
- **MPC-PureV64 ＋ `SAFE_BND` 2段カスケード**：浅読み MPC プローブ葉では無条件で Stage 1 (`V_1`) を返却し、本探索の窓近傍葉のみ Stage 2.5（16次元大域機動力・幾何線形、~15%）→ Stage 3（20ブロック偶奇分離 Key-14 SCReLU ＋ 60手番 `24→16→1` 純整数 MLP、~5%）を起動。
- **既存成果物を完全保護**：既存 `cache_bin/{train.bin, val.bin}` と `run_egaroucid_25m_v20/raw_ntuple64.p2c` は読み取り再利用のみ。v3.0 の全成果物は新規ディレクトリ `run_egaroucid_25m_v30/` へ保存します。

## Cell 1: GPU/VRAM 自動判定 ＆ Googleドライブマウント

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "run_egaroucid_25m_v30"
V20_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v20"
DRIVE_DATA_DIR = "/content/drive/MyDrive/p2c64_data"
DRIVE_CACHE_DIR = f"{DRIVE_DATA_DIR}/cache_bin"
DRIVE_WORK_DIR = f"/content/drive/MyDrive/p2c64_runs/{RUN_NAME}"
LOCAL_DATA_DIR = "/content/data"
REPO_DIR = "/content/p2c64-othello"

os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

VRAM_GB = 0.0
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU     : {torch.cuda.get_device_name(0)} ({VRAM_GB:.1f} GiB VRAM)")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
else:
    print("Running on CPU runtime (recommended for Cell 6 pure-scalar 1,000-game matches & MPC sigma)")

TRAIN_BATCH = 32768 if VRAM_GB >= 20.0 else 16384
MLP_HIDDEN = 16  # 16 (default, fastest pure-scalar) or 24
print(f"Config  : RUN_NAME={RUN_NAME}, TRAIN_BATCH={TRAIN_BATCH}, MLP_HIDDEN={MLP_HIDDEN}")
!lscpu | grep -E "Model name|CPU\(s\):|L1d|L2|L3" | head -n 10

## Cell 2: リポジトリ `clone` / `pull` ＆ C++ 純スカラービルド・単体テスト・0-SIMD ASM監査

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/Taichi111009/p2c64-othello.git"
REPO_DIR="/content/p2c64-othello"

if [ -d "$REPO_DIR/.git" ]; then
    cd "$REPO_DIR" && git fetch origin && git pull --ff-only origin main
else
    git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
make clean all test audit
ls -lh build/

## Cell 3: 既存キャッシュ `train.bin` / `val.bin` および凍結 Phase-1 `raw_ntuple64.p2c` の再利用準備

In [ ]:
import os
import shutil
import subprocess

cache_tr = f"{DRIVE_CACHE_DIR}/train.bin"
cache_va = f"{DRIVE_CACHE_DIR}/val.bin"
local_tr = f"{LOCAL_DATA_DIR}/train.bin"
local_va = f"{LOCAL_DATA_DIR}/val.bin"

for src, dst in [(cache_tr, local_tr), (cache_va, local_va)]:
    if os.path.exists(dst):
        print("[OK] Local SSD cache present:", dst)
    else:
        assert os.path.exists(src), f"Required binary cache not found on Drive: {src}"
        print(f"[Cache Hit] Copying {src} -> {dst} ...")
        shutil.copy2(src, dst)

# Reuse frozen Phase-1 baseline from run_egaroucid_25m_v20 (0-min Phase 1!)
v20_raw = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
v30_raw = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
local_raw = f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c"

if os.path.exists(v30_raw):
    print("[OK] Reusing raw_ntuple64.p2c in v3.0 work dir:", v30_raw)
elif os.path.exists(v20_raw):
    print("[Reuse v2.0 Phase-1] Copying frozen raw_ntuple64.p2c -> v3.0 work dir (0-min Phase 1!)...")
    shutil.copy2(v20_raw, v30_raw)
else:
    print("[Fallback] v2.0 raw_ntuple64.p2c not found; running Phase 1 once...")
    subprocess.run([
        "python3", f"{REPO_DIR}/python/train_p2c64.py",
        "--train-bin", local_tr,
        "--val-bin", local_va,
        "--work-dir", DRIVE_WORK_DIR,
        "--phases", "phase1",
        "--resume",
        "--batch-size", str(TRAIN_BATCH),
        "--p1-epochs", "6",
    ], check=True)

shutil.copy2(v30_raw, local_raw)
!ls -lh /content/data/

## Cell 4: WACA-G2 v3.0 学習（Stage 2.5 → 凍結 → Stage 3: 20-Block EO Key-14 SCReLU + 24→H→1 MLP + QAT）

In [ ]:
!python3 {REPO_DIR}/python/train_waca_g2.py \
    --train-bin {LOCAL_DATA_DIR}/train.bin \
    --val-bin {LOCAL_DATA_DIR}/val.bin \
    --raw-p2c {LOCAL_DATA_DIR}/raw_ntuple64.p2c \
    --work-dir {DRIVE_WORK_DIR} \
    --mlp-hidden {MLP_HIDDEN} \
    --batch-size {TRAIN_BATCH} \
    --s25-epochs 3 \
    --s25-lr 2e-2 \
    --s25-smooth 1e-3 \
    --s3-epochs 6 \
    --s3-qat-epochs 2 \
    --s3-emb-lr 8e-3 \
    --s3-mlp-lr 4e-3 \
    --s3-ply-smooth 8e-4 \
    --s3-blk-smooth 4e-3 \
    --resume

print("\nSaved v3.0 artifacts on Google Drive:")
!ls -lh {DRIVE_WORK_DIR}/*.p2c {DRIVE_WORK_DIR}/metrics_v30.*

## Cell 5: C++/Python 全3段ビット完全一致（`S1=0 S2.5=0 S3=0`）＆ 16対称（$D_4 \times \mathbb{Z}_2$）監査

In [ ]:
!python3 {REPO_DIR}/python/test_bit_exact_io.py \
    --data {LOCAL_DATA_DIR}/val.bin \
    --num-pos 1000 \
    --models {DRIVE_WORK_DIR}/raw_ntuple64.p2c {DRIVE_WORK_DIR}/waca_g2.p2c \
    --bench-bridge

## Cell 6: （CPUランタイム推奨）完全公平1,000局ペア対局（Pure PVS / Fair Std-MPC / ゲート掃引）＆ MPC $\sigma$ 計測

GPUを切断し CPU ランタイムへ切り替えた後、Cell 1〜2 を再実行してから本セルを実行してください。以下をすべて自動計測して `run_egaroucid_25m_v30/` へ保存します：
1. **MPC $\sigma$ 計測**（`d2→d6`・`d4→d8`, 2,000局面）：`RAW_NTUPLE_64`、`WACA_G2`、`WACA_G2[MPC-PureV64:V1->G2]` の3系列を比較。
2. **Pure PVS（`--no-mpc`）1,000局ペア対局**（`20ms/手`, 256 MiB TT）：`WACA_G2` vs `RAW_NTUPLE_64`。
3. **Fair Std-MPC（`--mpc`, 双方同一標準MPC）1,000局ペア対局**（`20ms/手`, 256 MiB TT）：3組のゲート閾値 `(M_25, M_3) ∈ {(2.5, 1.5), (3.0, 1.6), (2.0, 1.2)}` で掃引。

In [ ]:
import os
import subprocess

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
MPC_BIN = f"{REPO_DIR}/build/measure_mpc_sigma"
RAW_P2C = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
WACA_P2C = f"{DRIVE_WORK_DIR}/waca_g2.p2c"
val_bin = f"{LOCAL_DATA_DIR}/val.bin" if os.path.exists(f"{LOCAL_DATA_DIR}/val.bin") else f"{DRIVE_CACHE_DIR}/val.bin"

# 1) MPC sigma measurement (includes WACA_G2 and MPC-PureV64: V1(d_s) -> WACA_G2(d_t)!)
for sh, dp in [(2, 6), (4, 8)]:
    print(f"\n=== P2C-64 v3.0 MPC Sigma: depth {sh} -> {dp} ===")
    subprocess.run([
        MPC_BIN,
        "--data", val_bin,
        "--num-pos", "2000",
        "--shallow", str(sh),
        "--deep", str(dp),
        "--tt-mb", "64",
        "--m25", "2.5",
        "--m3", "1.5",
        "--gate-mode", "safe_bnd",
        "--csv", f"{DRIVE_WORK_DIR}/mpc_sigma_v30_d{sh}_d{dp}.csv",
        "--models", RAW_P2C, WACA_P2C,
    ], check=True)

# 2) Pure PVS (--no-mpc) 1,000-game paired match (20ms/move, 256 MiB TT)
match_csv = f"{DRIVE_WORK_DIR}/match_results_v30.csv"
print("\n=== [Match 1] Pure PVS (--no-mpc, 20ms, 256MiB TT, 1000 games): WACA_G2 vs RAW_NTUPLE_64 ===")
subprocess.run([
    MATCH_BIN,
    "--p1", WACA_P2C,
    "--p2", RAW_P2C,
    "--mode", "time",
    "--limit", "20",
    "--pairs", "500",
    "--opening-plies", "8",
    "--endgame-empties", "10",
    "--tt-mb", "256",
    "--no-mpc",
    "--m25", "2.5",
    "--m3", "1.5",
    "--gate-mode", "safe_bnd",
    "--seed", "20261126",
    "--csv", match_csv,
], check=True)

# 3) Fair Std-MPC (--mpc, BOTH players use identical Std-MPC) 1,000-game matches + gate sweep
for m25, m3 in [("2.5", "1.5"), ("3.0", "1.6"), ("2.0", "1.2")]:
    print(f"\n=== [Fair Std-MPC, 20ms, 256MiB TT, 1000 games] WACA_G2 (M25={m25}, M3={m3}) vs RAW_NTUPLE_64 ===")
    subprocess.run([
        MATCH_BIN,
        "--p1", WACA_P2C,
        "--p2", RAW_P2C,
        "--mode", "time",
        "--limit", "20",
        "--pairs", "500",
        "--opening-plies", "8",
        "--endgame-empties", "10",
        "--tt-mb", "256",
        "--mpc",
        "--mpc-t", "1.5",
        "--m25", m25,
        "--m3", m3,
        "--gate-mode", "safe_bnd",
        "--seed", "20261126",
        "--csv", match_csv,
    ], check=True)

print(f"\nAll v3.0 match & MPC results saved to {DRIVE_WORK_DIR}")